Importing modules

In [1]:
import numpy as np
import matplotlib.pyplot as plt

Defining parameters

In [ ]:


n_users = 35        # secondary users
n_primary = 8       # primary channels
n_channels = 40     # available spectrum bands

noise = 0.1         # noise for SINR

PS_penalty = 75     # penalty for primary-econdary user channel collision
SS_penalty = 25     # penalty for secondary-secondary user channel collision

N=200                # particles
S=n_channels-1      # search space [0, n_channels-1]
D=n_users           # dimensions = number of users
n_iter = 600        # number of updates of positions

a=0.7
b=1.5
b_hat=1.5
c=0.1

beta_start = 1
beta_end = 0.5




Getting SINR from random

In [3]:
# Simulate channel gains randomly (in reality this comes from pathloss model)
# SINR[i][j] = signal quality if user i uses channel j
np.random.seed(67)
channel_gain = np.random.uniform(0.1, 1.0, (n_users, n_channels))     # to get SINR which gets throughput
SINR = channel_gain / noise                                           # simplified, no inter-user interference yet



Setting up Primary Users

In [4]:
# Primary user occupancy: PU[j] = 1 means channel j is occupied by primary user

PU_occupied = np.zeros(n_channels, dtype=int)
PU_occupied[np.random.choice(n_channels, n_primary, replace=False)] = 1

Fitness function which gets the throughput using SINR and adds penalty

In [5]:
def fitness(x, PS_penalty=PS_penalty, SS_penalty=SS_penalty):
    channels = np.clip(np.round(x).astype(int), 0, n_channels - 1)
    
    pu_mask = PU_occupied[channels]                          # 1 where PU is present
    sinr_vals = SINR[np.arange(n_users), channels]          # SINR for each user's channel
    
    throughput = np.sum(np.log2(1 + sinr_vals) * (1 - pu_mask))
    ps_penalty = np.sum(pu_mask) * PS_penalty
    
    counts = np.bincount(channels, minlength=n_channels)
    ss_pen = np.sum(np.maximum(counts - 1, 0)) * SS_penalty
    
    return -throughput + ps_penalty + ss_pen

PSO algorithm

In [6]:
def dpso(f, D, N, S, n_iter, a, b, b_hat, c, see):
    np.random.seed(see)
    x = np.random.uniform(0, S, size=(N, D))                 # setting up the initial positions of the N number of particles
    v = np.random.normal(size=(N, D))                        # setting up the initial velocities
    p = x.copy()                                             # best particle position
    fp = np.array([f(p[i]) for i in range(N)])               # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                          # global best position
    fp_hat = f(p_hat)                                        # throughput of global best position
    fp_hat_his = []

    for i in range (n_iter):
        fp_hat_his.append(float(fp_hat))
        #if i%(n_iter//10)== 0:                               # to show progress
            #print(f"progress {(i/n_iter)*100:.0f}%")
            #pass
            

        r,r_hat = np.random.uniform(0, 1, (2,N, D))          # setting up random parameters

        v = a*v + b*r*(p-x) + b_hat*r_hat*(p_hat-x)          # updating velocities as vector sum of the directions of initial velocity, local minima, local maxima
        x = x + c*v                                          # updating position according to velocities
        x = np.clip(x, 0, S)                                 # to limit the range within the subspace

        for n in range(N):                                   # calculation for each particle

            xn = x[n]                                        # getting position of that particle           
            fxn = f(xn)                                      # current throughput of that particle
            fpn = fp[n]                                      # best throughput of that particle

            if fxn < fpn:                                    # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                             # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    # print("progress 100%")
    return p_hat,fp_hat_his                                  # "coordinates", ie channel allocation of global best throughput

        



Calling PSO 

In [ ]:
n_seeds = 30
n_parameter = 800 

results = np.empty((n_seeds, n_parameter//10))
for s in range(n_seeds):
    print("seed", s)
    for ss_idx, SS in enumerate(range(0,n_parameter,10)):
        result, history = dpso(fitness, D, N, S, SS+1, a, b, b_hat, c,s)
        C_best_assignment = np.clip(np.round(result).astype(int), 0, n_channels-1)
        C_throughput = 0
        for user in range(n_users):
            ch = C_best_assignment[user]
            if not PU_occupied[ch]:
                C_throughput += np.log2(1 + SINR[user, ch])
        results[s, ss_idx] = C_throughput   




avg_throughput = np.mean(results, axis=0)



plt.plot(range(n_parameter//10), avg_throughput)
plt.xlabel("Number of iterations")
plt.ylabel("Avg Throughput (over 30 seeds)")
plt.title("PSO — Number of iterations vs Throughput")
plt.show() 

seed 0


KeyboardInterrupt: 

QPSO algorithm

In [ ]:
def dqpso(f, D, N, S, n_iter, beta_start, beta_end):

    x = np.random.uniform(0, S, size=(N, D))                            # setting up the initial positions of the N number of particles
    p = x.copy()                                                        # best particle position
    fp = np.array([f(p[i]) for i in range(N)])                          # throughput of all particles
    p_hat = x[np.argmin(fp)].copy()                                     # global best position
    fp_hat = f(p_hat)                                                   # throughput of global best position


    for i in range(n_iter):

        if i%(n_iter//10)== 0:                                          # to show progress
            print(f"progress {(i/n_iter)*100:.0f}%")
        
        
        beta = beta_start - (beta_start - beta_end) * i / n_iter        # Beta decreases linearly from beta_start to beta_end

        mbest = np.mean(p, axis=0)                                      # Mean best position ie average of all personal bests

        phi = np.random.uniform(0,1, (N,D))                             
        p_local = phi * p + (1 - phi) * p_hat                           # local attractor for each particle (works like velocity or inertia)

        u = np.random.uniform(1e-12, 1, (N,D))                           
        sign = 2 * np.random.randint(0, 2, size=(N,D)) - 1
        x = p_local + sign * beta * np.abs(mbest - x) * np.log(1/u)     # calculates the next value of x
        x = np.clip(x, 0, S)                                            # to limit the range within the subspace


        for n in range(N):                                              # calculation for each particle

            xn = x[n]                                                   # getting position of that particle           
            fxn = f(xn)                                                 # current throughput of that particle
            fpn = fp[n]                                                 # best throughput of that particle

            if fxn < fpn:                                               # if current throughput of that particle is better the previous ones, update
                p[n] = xn.copy()
                fp[n] = fxn

                if fxn < fp_hat:                                        # if the current througput is the global best throughput, update
                    p_hat = xn.copy()
                    fp_hat = fxn
    
    print("progress 100%")
    return p_hat                                                        # "coordinates", ie channel allocation of global best throughput

            

    

Calling QPSO

In [ ]:
"""
result_qpso = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end)

Q_best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)

Q_throughput = 0
for user in range(n_users):
    ch = Q_best_assignment[user]
    if not PU_occupied[ch]:
        Q_throughput += np.log2(1 + SINR[user, ch])
"""



'\nresult_qpso = dqpso(fitness, D, N, S, n_iter, beta_start, beta_end)\n\nQ_best_assignment = np.clip(np.round(result_qpso).astype(int), 0, n_channels-1)\n\nQ_throughput = 0\nfor user in range(n_users):\n    ch = Q_best_assignment[user]\n    if not PU_occupied[ch]:\n        Q_throughput += np.log2(1 + SINR[user, ch])\n'

In [ ]:
print("Classical Particle Swarm Optimization\n")
print("Channel assignment:", C_best_assignment)
print("Throughput:", C_throughput)
'''
print("\n\nQuantum Particle Swarm Optimization \n")
print("Channel assignment:", Q_best_assignment)
print("Throughput:", Q_throughput)
'''

Classical Particle Swarm Optimization

Channel assignment: [11 26 17 22 37 37 30 14 34 26 16  1 30 22 26 28 29 12 31 21 19 26  4  4
 15 38  6 13  2 14 11 34 29 15  2]
Throughput: 95.42887235628508


'\nprint("\n\nQuantum Particle Swarm Optimization \n")\nprint("Channel assignment:", Q_best_assignment)\nprint("Throughput:", Q_throughput)\n'